In [1]:
!nvidia-smi
!pip install ultralytics -q
import ultralytics
ultralytics.checks()

Ultralytics 8.4.118 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 46.9/112.6 GB disk)


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
yaml_content = """
path: /content/drive/MyDrive/UCT_CropWeed/CropWeedDataset
train: images/train
val: images/val

nc: 2
names: ['crop', 'weed']
"""

with open("/content/drive/MyDrive/UCT_CropWeed/CropWeedDataset/data.yaml", "w") as f:
    f.write(yaml_content.strip())

print("data.yaml updated successfully!")

# Verify
!cat /content/drive/MyDrive/UCT_CropWeed/CropWeedDataset/data.yaml

data.yaml updated successfully!
path: /content/drive/MyDrive/UCT_CropWeed/CropWeedDataset
train: images/train
val: images/val

nc: 2
names: ['crop', 'weed']

In [8]:
import os

# Change this path if your folder name is slightly different
DATA_PATH = "/content/drive/MyDrive/UCT_CropWeed/CropWeedDataset"
YAML_PATH = f"{DATA_PATH}/data.yaml"

print("Files in dataset folder:")
print(os.listdir(DATA_PATH))

# Show the yaml content
!cat {YAML_PATH}

Files in dataset folder:
['labels', 'images', 'data.yaml']
path: /content/drive/MyDrive/UCT_CropWeed/CropWeedDataset
train: images/train
val: images/val

nc: 2
names: ['crop', 'weed']

In [9]:
from ultralytics import YOLO

# Load a small pretrained model (good starting point)
model = YOLO("yolov8n.pt")   # you can also try yolov8s.pt later

# Train
results = model.train(
    data=YAML_PATH,
    epochs=50,          # start with 50. Increase to 80-100 if you have time
    imgsz=512,
    batch=16,           # reduce to 8 if you get out-of-memory error
    name="crop_weed_yolov8",
    project="/content/drive/MyDrive/UCT_CropWeed/runs",
    exist_ok=True,
    patience=15,        # early stopping
    save=True,
    plots=True
)

Ultralytics 8.4.118 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/UCT_CropWeed/CropWeedDataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=c

In [10]:
# Load the best model that was saved
best_model = YOLO("/content/drive/MyDrive/UCT_CropWeed/runs/crop_weed_yolov8/weights/best.pt")

metrics = best_model.val()
print(metrics)

Ultralytics 8.4.118 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.7±0.4 ms, read: 42.8±22.3 MB/s, size: 70.6 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/drive/MyDrive/UCT_CropWeed/CropWeedDataset/labels/val.cache... 260 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 260/260 99.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 3.7it/s 4.6s
                   all        260        415      0.781      0.804      0.843      0.567
                  crop        121        241      0.693      0.793       0.83      0.572
                  weed        139        174      0.869      0.816      0.856      0.561
Speed: 2.8ms preprocess, 3.5ms inference

In [11]:
# Predict on a few images and save results
results = best_model.predict(
    source=f"{DATA_PATH}/images/val",
    conf=0.25,
    save=True,
    project="/content/drive/MyDrive/UCT_CropWeed/predictions",
    name="val_predictions",
    exist_ok=True
)

print("Predictions saved!")



image 1/260 /content/drive/MyDrive/UCT_CropWeed/CropWeedDataset/images/val/agri_0_1095.jpeg: 512x512 1 weed, 6.2ms
image 2/260 /content/drive/MyDrive/UCT_CropWeed/CropWeedDataset/images/val/agri_0_1123.jpeg: 512x512 3 crops, 6.8ms
image 3/260 /content/drive/MyDrive/UCT_CropWeed/CropWeedDataset/images/val/agri_0_113.jpeg: 512x512 1 weed, 8.7ms
image 4/260 /content/drive/MyDrive/UCT_CropWeed/CropWeedDataset/images/val/agri_0_1166.jpeg: 512x512 1 weed, 6.7ms
image 5/260 /content/drive/MyDrive/UCT_CropWeed/CropWeedDataset/images/val/agri_0_1173.jpeg: 512x512 1 weed, 6.1ms
image 6/260 /content/drive/MyDrive/UCT_CropWeed/CropWeedDataset/images/val/agri_0_1177.jpeg: 512x512 1 crop, 6.2ms
image 7/260 /content/drive/MyDrive/UCT_CropWeed/CropWeedDataset/images/val/agri_0_1211.jpeg: 512x512 1 crop, 6.5ms
image 8/260 /content/drive/MyDrive/UCT_CropWeed/CropWeedDataset/images/val/agri_0_126.jpeg: 512x512 6 weeds, 6.0ms
image 9/260 /content/drive/MyDrive/UCT_CropWeed/CropWeedDataset/images/val/agri

In [12]:
from google.colab import files

# Download the best weights
files.download("/content/drive/MyDrive/UCT_CropWeed/runs/crop_weed_yolov8/weights/best.pt")

# You can also zip the whole runs folder if you want

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>